# South Africa vs. Lesotho: Price Series and Inflation Comparison

Lesotho's loti is pegged 1:1 to the South African rand under the Common Monetary Area (CMA), so the two countries share an exchange rate against the USD. Inflation, however, is set domestically. This notebook compares the consumer price index (`NECONPRVTXN`) for ZAF and LSO from the raw MFMOD extract, restricted to the historical window (1980-2023).

In [ ]:
import pandas as pd
import numpy as np
import altair as alt

alt.data_transformers.disable_max_rows()

df = pd.read_csv('../data/priceIndex_and_exchangeRate_series_2025_SP.csv', index_col=0)
df.index = pd.to_datetime(df.index)
df = df[(df.index >= '1980-01-01') & (df.index <= '2023-01-01')]

prices = pd.DataFrame({
    'ZAF': df['ZAFNECONPRVTXN'],
    'LSO': df['LSONECONPRVTXN'],
})
prices.index.name = 'date'
prices.head()

## Data coverage

In [ ]:
coverage = pd.DataFrame({
    'first_valid': [prices[c].first_valid_index() for c in prices.columns],
    'last_valid':  [prices[c].last_valid_index()  for c in prices.columns],
    'n_obs':       [prices[c].notna().sum()       for c in prices.columns],
}, index=prices.columns)
coverage

## Rebase to 1980 = 100 and plot the price level

The raw series is an index normalized to 1 in some reference year (2012 for LSO, 2015 for ZAF). Rebasing both to 100 in 1980 makes cumulative inflation directly comparable.

In [ ]:
rebased = prices.divide(prices.iloc[0]).multiply(100)

tidy = (rebased
        .reset_index()
        .melt('date', var_name='country', value_name='cpi_rebased'))

alt.Chart(tidy).mark_line().encode(
    x=alt.X('year(date):T', title='Year'),
    y=alt.Y('cpi_rebased:Q', title='CPI (1980 = 100)'),
    color=alt.Color('country:N', title='Country')
).properties(
    width=700, height=350,
    title='Consumer price index, ZAF vs LSO (1980 = 100)'
)

## Annual inflation (year-on-year % change)

In [ ]:
inflation = prices.pct_change().multiply(100)
inflation.index.name = 'date'

tidy_inf = (inflation
            .reset_index()
            .melt('date', var_name='country', value_name='inflation_pct')
            .dropna())

alt.Chart(tidy_inf).mark_line().encode(
    x=alt.X('year(date):T', title='Year'),
    y=alt.Y('inflation_pct:Q', title='YoY inflation (%)'),
    color=alt.Color('country:N', title='Country')
).properties(
    width=700, height=350,
    title='Annual CPI inflation, ZAF vs LSO'
)

## Inflation differential (LSO minus ZAF)

Under the CMA peg, a persistent positive gap implies real appreciation of the loti against the rand.

In [ ]:
diff = (inflation['LSO'] - inflation['ZAF']).rename('lso_minus_zaf').reset_index().dropna()

alt.Chart(diff).mark_bar().encode(
    x=alt.X('year(date):T', title='Year'),
    y=alt.Y('lso_minus_zaf:Q', title='LSO inflation - ZAF inflation (pp)'),
    color=alt.condition('datum.lso_minus_zaf >= 0',
                        alt.value('#d62728'),
                        alt.value('#1f77b4'))
).properties(
    width=700, height=300,
    title='Inflation differential: LSO minus ZAF (percentage points)'
)

## Summary statistics

In [ ]:
def geo_mean_inflation(series):
    s = series.dropna() / 100 + 1
    return (s.prod() ** (1 / len(s)) - 1) * 100

summary = pd.DataFrame({
    'mean_inflation_pct':        inflation.mean(),
    'median_inflation_pct':      inflation.median(),
    'geo_mean_inflation_pct':    inflation.apply(geo_mean_inflation),
    'std_inflation_pct':         inflation.std(),
    'min_inflation_pct':         inflation.min(),
    'max_inflation_pct':         inflation.max(),
    'cumulative_price_growth_x': prices.iloc[-1] / prices.iloc[0],
})
summary.round(2)

In [ ]:
# Decade-average inflation
by_decade = inflation.copy()
by_decade['decade'] = (by_decade.index.year // 10) * 10
by_decade.groupby('decade')[['ZAF', 'LSO']].mean().round(2)

## Reading the comparison (preliminary — see WDI check below)

From the MFMOD extract alone:

- **Cumulative inflation**: South Africa's price level grows by roughly an order of magnitude more than Lesotho's between 1980 and 2023 (see `cumulative_price_growth_x`). On its face that puts ZAF geometric-mean inflation around 8-9% per year against ~2-3% for LSO.
- **Anomaly in the LSO series**: that result is hard to reconcile with the Common Monetary Area peg, which would normally push the two inflation rates together. The LSO price index in this extract also shows ~16% year-on-year *deflation* in 1982 and a sustained drop from 1981 through 1984 — implausible for a rand-pegged economy during a period of rising South African prices.

The next section cross-checks both series against World Bank WDI to determine whether MFMOD has a data error.

## Cross-check against World Bank WDI

The discrepancy is large enough to warrant external validation. The WDI series `FP.CPI.TOTL.ZG` (inflation, consumer prices, annual %) is the standard reference. The table below was pulled from `api.worldbank.org` for both countries and is hard-coded here so the comparison can be re-run without network access.

In [ ]:
# World Bank WDI: inflation, consumer prices (annual %), FP.CPI.TOTL.ZG
# Source: https://api.worldbank.org/v2/country/{ISO}/indicator/FP.CPI.TOTL.ZG
wdi_zaf = {
    1980: 13.66, 1981: 15.25, 1982: 14.64, 1983: 12.30, 1984: 11.53,
    1985: 16.29, 1986: 18.65, 1987: 16.16, 1988: 12.78, 1989: 14.73,
    1990: 14.32, 1991: 15.33, 1992: 13.87, 1993: 9.72,  1994: 8.94,
    1995: 8.68,  1996: 7.35,  1997: 8.60,  1998: 6.88,  1999: 5.18,
    2000: 5.34,  2001: 5.70,  2002: 9.49,  2003: 5.68,  2004: -0.69,
    2005: 2.06,  2006: 3.24,  2007: 6.18,  2008: 9.91,  2009: 7.24,
    2010: 4.07,  2011: 5.00,  2012: 5.74,  2013: 5.78,  2014: 6.13,
    2015: 4.52,  2016: 6.60,  2017: 5.19,  2018: 4.51,  2019: 4.10,
    2020: 3.23,  2021: 4.62,  2022: 7.04,  2023: 6.08,
}
wdi_lso = {
    1980: 16.27, 1981: 12.41, 1982: 12.14, 1983: 17.49, 1984: 10.98,
    1985: 13.32, 1986: 18.00, 1987: 11.75, 1988: 11.46, 1989: 14.73,
    1990: 11.63, 1991: 17.68, 1992: 17.21, 1993: 13.14, 1994: 8.21,
    1995: 9.27,  1996: 9.33,
    # 1997-1999: not reported by WDI
    2000: 6.13,  2001: -9.62, 2002: 33.81, 2003: 6.63,  2004: 5.02,
    2005: 3.44,  2006: 6.07,  2007: 8.01,  2008: 10.72, 2009: -16.86,
    2010: -2.41, 2011: 5.04,  2012: 6.05,  2013: 4.87,  2014: 5.37,
    2015: 3.22,  2016: 6.60,  2017: 4.45,  2018: 4.75,  2019: 5.19,
    2020: 4.98,  2021: 6.05,  2022: 8.27,  2023: 6.34,
}

wdi = pd.DataFrame({'ZAF': wdi_zaf, 'LSO': wdi_lso})
wdi.index.name = 'year'

# MFMOD-implied YoY inflation
mfmod = inflation.copy()
mfmod.index = mfmod.index.year
mfmod.index.name = 'year'

compare = (
    mfmod.join(wdi, lsuffix='_mfmod', rsuffix='_wdi')
         .reindex(columns=['ZAF_mfmod', 'ZAF_wdi', 'LSO_mfmod', 'LSO_wdi'])
)
compare.round(2)

In [ ]:
# Visualize MFMOD vs WDI for both countries
long = (compare
        .reset_index()
        .melt('year', var_name='series', value_name='inflation_pct')
        .dropna())
long[['country', 'source']] = long['series'].str.split('_', expand=True)

alt.Chart(long).mark_line(point=True).encode(
    x=alt.X('year:O', title='Year'),
    y=alt.Y('inflation_pct:Q', title='YoY inflation (%)'),
    color=alt.Color('source:N', title='Source'),
    strokeDash=alt.StrokeDash('source:N'),
).properties(width=700, height=250).facet(
    row=alt.Row('country:N', title=None)
).resolve_scale(y='independent').properties(
    title='MFMOD-implied vs World Bank WDI inflation'
)

In [ ]:
# Cumulative price-level comparison: MFMOD vs WDI-implied
def chain_from_inflation(rates_pct, start_year, base=100.0):
    s = pd.Series(rates_pct).sort_index()
    s = s.loc[s.index >= start_year]
    out = {}
    level = base
    out[s.index[0] - 1] = base
    for y, r in s.items():
        if pd.isna(r):
            level = np.nan
            out[y] = np.nan
            continue
        if pd.isna(level):
            # restart from last known level if any
            prev = pd.Series(out).dropna()
            level = prev.iloc[-1] if len(prev) else base
        level = level * (1 + r / 100)
        out[y] = level
    return pd.Series(out).sort_index()

# Anchor everything at 1980 = 100
wdi_zaf_level = chain_from_inflation(wdi['ZAF'], 1981, base=100.0)
wdi_lso_level = chain_from_inflation(wdi['LSO'], 1981, base=100.0)

mfmod_levels = rebased.copy()
mfmod_levels.index = mfmod_levels.index.year

levels = pd.DataFrame({
    'ZAF_mfmod': mfmod_levels['ZAF'],
    'ZAF_wdi':   wdi_zaf_level,
    'LSO_mfmod': mfmod_levels['LSO'],
    'LSO_wdi':   wdi_lso_level,
})
levels.tail(10).round(1)

## Verdict: MFMOD's LSO `NECONPRVTXN` series is wrong in the 1980s

**ZAF is fine.** Year-on-year inflation implied by `ZAFNECONPRVTXN` matches WDI to within ~1-3 pp across the full sample. Cumulative growth 1980-2023 also lines up (both around 40-45×).

**LSO is materially wrong, especially pre-1990.** WDI shows Lesotho running 11-18% inflation every year of the 1980s — broadly tracking South Africa's, as the CMA peg requires. The MFMOD series instead shows 0% in 1981, **-16% in 1982**, then several more years of deflation, with cumulative growth 1980-1996 of only ~1.4× against WDI-implied ~10×. This is a roughly factor-of-7 discrepancy and cannot be explained by base-year or unit choices.

**Likely cause.** The Lesotho `NECONPRVTXN` column in this MFMOD vintage appears to have been spliced from a non-CPI source for the early years — candidates include a private-consumption deflator in constant local currency, a real consumption volume index, or a back-cast that failed. Whatever the source, it is not the consumer price index it is labelled as.

**Implications for the wider analysis.**

- Any hybrid index `H = P/X` built from this column for Lesotho will be biased upward in real terms in the 1980s (because P is understated) and therefore overstate Lesotho's real depreciation relative to the rand in subsequent years.
- The event-detection routine in `data_analysis_functions.identify_events` is keyed off deviations of H from its 5-year moving average; a spurious early-1980s drop in P would propagate into a spurious early "recovery" in H and likely create false-negative event coverage in the 1980s and false positives later.
- Recommended remediation: either (a) replace `LSONECONPRVTXN` with WDI `FP.CPI.TOTL` (level series) before constructing H, or (b) drop Lesotho pre-1995 from the sample, or (c) splice WDI onto MFMOD at a credible join year (e.g., 2000) and document the splice.

**Answering the original question.** Once we use WDI rather than MFMOD as the source of truth, South African and Lesotho inflation are very similar in level over 1980-2023, as the CMA peg implies — both compounding at roughly 9-10% per year in the 1980s, then disinflating to mid-single digits from the mid-1990s onward. Lesotho's series is noisier (the WDI itself shows wild swings around 2001-02 and 2008-10 that likely reflect survey-methodology breaks rather than real macro events), but the two countries' long-run cumulative inflation is broadly comparable. The big divergence shown by the MFMOD extract alone is an artefact of the underlying data, not a real economic phenomenon.

**Sources.**
- World Bank WDI, `FP.CPI.TOTL.ZG` for [LSO](https://api.worldbank.org/v2/country/LSO/indicator/FP.CPI.TOTL.ZG?date=1980:2023&format=json) and [ZAF](https://api.worldbank.org/v2/country/ZAF/indicator/FP.CPI.TOTL.ZG?date=1980:2023&format=json).